# GuidaPlate — LSTM Dietary Pattern Analyzer
## Meal Sequence Training on NHANES 2017-2018 CKD Cohort

**Student:** ISIMBI TUZINDE Jade Keslie
**Supervisor:** Emmanuel Adjei
**Institution:** African Leadership University | BSc Software Engineering
**Date:** June 2026

### Research Context
GuidaPlate uses NHANES 2017-2018 as its training dataset because no publicly available Rwandan CKD dietary dataset exists that combines dietary recall with clinical laboratory results.

The LSTM detects patterns of cumulative nutrient threshold violations across meal occasions. These patterns are governed by universal KDOQI 2020 clinical thresholds which apply equally to Rwandan and American CKD patients.

The Rwanda-specific food database (50 foods, Kenya FCT 2018) is used in the recommendation engine deployed to Rwandan patients, not in model training.

### Why LSTM
XGBoost classifies daily totals. LSTM detects dangerous eating patterns across 6 meal occasions over 2 days.

Alsulami et al. 2025 demonstrated 96% accuracy using LSTM on dietary sequence data for chronic disease prediction. The gated memory architecture retains information across meal occasions to detect cumulative risk.

### Input Sequence
Each patient = 6 meal steps: Day 1 Breakfast → Lunch → Dinner; Day 2 Breakfast → Lunch → Dinner.

Each step = 4 nutrient values: [potassium, phosphorus, protein_per_kg, sodium]

Input shape: (n_patients, 6, 4)

In [ ]:
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay,
)
import joblib

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

try:
    plt.style.use('seaborn-v0_8')
except OSError:
    plt.style.use('seaborn')

%matplotlib inline

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

RANDOM_STATE = 42
TEST_SIZE = 0.2
ALPHA = 0.05

STAGE_ORDER = ['G2', 'G3a', 'G3b', 'G4']
RISK_CLASSES = ['LOW', 'MODERATE', 'HIGH']
RISK_ENCODE = {c: i for i, c in enumerate(RISK_CLASSES)}

MEAL_OCCASIONS = {
    1: 'D1_Breakfast', 2: 'D1_Lunch', 3: 'D1_Dinner',
    4: 'D2_Breakfast', 5: 'D2_Lunch', 6: 'D2_Dinner',
}

KDOQI = {
    'G2':  {'potassium': 3500, 'phosphorus': 1000, 'protein_per_kg': 0.8, 'sodium': 2300},
    'G3a': {'potassium': 3000, 'phosphorus': 800,  'protein_per_kg': 0.6, 'sodium': 2300},
    'G3b': {'potassium': 3000, 'phosphorus': 800,  'protein_per_kg': 0.6, 'sodium': 2300},
    'G4':  {'potassium': 2500, 'phosphorus': 700,  'protein_per_kg': 0.55, 'sodium': 2300},
}

def project_root():
    p = Path('.').resolve()
    if p.name == 'notebooks':
        return p.parent
    if (p / 'data').exists():
        return p
    return p.parent

ROOT = project_root()
FIG_DIR = ROOT / 'outputs' / 'figures'
STATS_DIR = ROOT / 'outputs' / 'stats'
MODEL_DIR = ROOT / 'models'
FIG_DIR.mkdir(parents=True, exist_ok=True)
STATS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {ROOT}")

## Section 3 — Load Individual Food Files

NHANES DR1IFF and DR2IFF contain every individual food item each participant ate on Day 1 and Day 2.

Key columns used:
- SEQN: participant ID
- DR1ILINE/DR2ILINE: food line number
- DR1DRSTZ: dietary recall status
- DR1_020/DR2_020: meal occasion code (1=Breakfast, 2=Lunch, 3=Dinner, 4=Snack, 5=Drink, 6=Infant)
- DR1IPOTA/DR2IPOTA: potassium (mg)
- DR1IPHOS/DR2IPHOS: phosphorus (mg)
- DR1IPROT/DR2IPROT: protein (g)
- DR1ISODI/DR2ISODI: sodium (mg)

In [ ]:
def load_iff(path: Path) -> pd.DataFrame:
    try:
        import pyreadstat
        df, _ = pyreadstat.read_xpt(str(path))
        print(f"Loaded {path.name} via pyreadstat")
        return df
    except Exception as exc:
        print(f"pyreadstat unavailable ({exc}); using pandas.read_sas")
        return pd.read_sas(path, format='xport')


def pick_col(df: pd.DataFrame, candidates: list[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    for c in candidates:
        alt = c.replace('.', '_')
        if alt in df.columns:
            return alt
    matches = [col for col in df.columns if any(c.replace('.', '_') in str(col) for c in candidates)]
    if matches:
        return matches[0]
    raise KeyError(f"None of {candidates} found. Available sample: {list(df.columns)[:20]}")


def standardize_iff(df: pd.DataFrame, day: int) -> pd.DataFrame:
    prefix = f'DR{day}'
    rename = {
        pick_col(df, [f'{prefix}.020', f'{prefix}_020']): 'meal_code',
        pick_col(df, [f'{prefix}IPOTA']): 'potassium',
        pick_col(df, [f'{prefix}IPHOS']): 'phosphorus',
        pick_col(df, [f'{prefix}IPROT']): 'protein',
        pick_col(df, [f'{prefix}ISODI']): 'sodium',
    }
    out = df.rename(columns=rename)
    return out[['SEQN', 'meal_code', 'potassium', 'phosphorus', 'protein', 'sodium']].copy()


print("Loading DR1IFF...")
iff1_path = ROOT / 'data' / 'raw' / 'nhanes' / 'DR1IFF_J.xpt'
iff1 = load_iff(iff1_path)
print(f"DR1IFF shape: {iff1.shape}")

print("Loading DR2IFF...")
iff2_path = ROOT / 'data' / 'raw' / 'nhanes' / 'DR2IFF_J.xpt'
iff2 = load_iff(iff2_path)
print(f"DR2IFF shape: {iff2.shape}")

print("Loading CKD cohort...")
cohort = pd.read_csv(ROOT / 'data' / 'processed' / 'ckd_cohort_final.csv')
print(f"CKD cohort: {cohort.shape}")

labels = pd.read_csv(ROOT / 'outputs' / 'stats' / '05_risk_labels.csv')
print(f"Risk labels: {labels.shape}")

cohort = cohort.merge(labels[['SEQN', 'risk_label']], on='SEQN', how='inner')
print(f"Cohort with labels: {cohort.shape}")
print()
print("CKD patients in cohort:")
print(cohort['ckd_stage'].value_counts().reindex(STAGE_ORDER))

## Section 4 — Build Meal Sequences

For each CKD patient build a sequence of 6 meal occasions by:

1. Filtering IFF files to only CKD cohort patients
2. Mapping meal occasion codes to 6 standard slots
3. Summing nutrients per meal slot
4. Filling missing meals with zeros
5. Normalizing protein to g/kg using patient body weight

In [ ]:
ckd_seqns = set(cohort['SEQN'])
print(f"CKD patients to process: {len(ckd_seqns)}")


import datetime

def map_meal_slot(day, meal_code):
    # Map DR1_020/DR2_020 (occasion code or seconds since midnight) to 6 meal slots.
    if pd.isna(meal_code):
        return 2 if day == 1 else 5
    if isinstance(meal_code, datetime.time):
        code = meal_code.hour * 3600 + meal_code.minute * 60 + meal_code.second
    else:
        try:
            code = float(meal_code)
        except (TypeError, ValueError):
            return 2 if day == 1 else 5
    if code <= 10:
        if code == 1:
            return 0 if day == 1 else 3
        if code == 2:
            return 1 if day == 1 else 4
        return 2 if day == 1 else 5
    # Time-of-day in seconds since midnight
    if code < 39600:   # before 11:00
        return 0 if day == 1 else 3
    if code < 61200:   # before 17:00
        return 1 if day == 1 else 4
    return 2 if day == 1 else 5


print("Processing Day 1 foods...")
iff1_ckd = standardize_iff(iff1[iff1['SEQN'].isin(ckd_seqns)].copy(), day=1)
iff1_ckd['meal_slot'] = iff1_ckd['meal_code'].apply(lambda x: map_meal_slot(1, x))
iff1_ckd['day'] = 1
print(f"Day 1 food records for CKD patients: {len(iff1_ckd)}")

print("Processing Day 2 foods...")
iff2_ckd = standardize_iff(iff2[iff2['SEQN'].isin(ckd_seqns)].copy(), day=2)
iff2_ckd['meal_slot'] = iff2_ckd['meal_code'].apply(lambda x: map_meal_slot(2, x))
iff2_ckd['day'] = 2
print(f"Day 2 food records for CKD patients: {len(iff2_ckd)}")

all_foods = pd.concat([
    iff1_ckd[['SEQN', 'meal_slot', 'potassium', 'phosphorus', 'protein', 'sodium']],
    iff2_ckd[['SEQN', 'meal_slot', 'potassium', 'phosphorus', 'protein', 'sodium']],
], ignore_index=True)

for col in ['potassium', 'phosphorus', 'protein', 'sodium']:
    all_foods[col] = pd.to_numeric(all_foods[col], errors='coerce').fillna(0)

meal_nutrients = all_foods.groupby(['SEQN', 'meal_slot'])[['potassium', 'phosphorus', 'protein', 'sodium']].sum().reset_index()
print()
print(f"Meal nutrient records: {len(meal_nutrients)}")
print()
print("Building sequence arrays...")
sequence_data = []
sequence_labels = []
sequence_seqns = []

cohort_weights = cohort[['SEQN', 'weight_kg', 'ckd_stage', 'risk_label']].copy()
n_processed = 0
n_skipped = 0

for _, patient in cohort_weights.iterrows():
    seqn = patient['SEQN']
    weight = patient['weight_kg']
    risk = patient['risk_label']

    if pd.isna(weight) or weight <= 0:
        n_skipped += 1
        continue
    if risk not in RISK_CLASSES:
        n_skipped += 1
        continue

    patient_meals = meal_nutrients[meal_nutrients['SEQN'] == seqn]
    seq = np.zeros((6, 4))

    for _, meal in patient_meals.iterrows():
        slot = int(meal['meal_slot'])
        if 0 <= slot <= 5:
            seq[slot, 0] = meal['potassium']
            seq[slot, 1] = meal['phosphorus']
            seq[slot, 2] = meal['protein'] / weight
            seq[slot, 3] = meal['sodium']

    sequence_data.append(seq)
    sequence_labels.append(risk)
    sequence_seqns.append(seqn)
    n_processed += 1

X_seq = np.array(sequence_data)
y_seq = np.array(sequence_labels)

print(f"Sequences built: {n_processed}")
print(f"Skipped: {n_skipped}")
print(f"Sequence array shape: {X_seq.shape}")
print()
print("Risk label distribution:")
unique, counts = np.unique(y_seq, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {u}: {c} ({c/len(y_seq)*100:.1f}%)")

# Cache sequences and labels so other notebooks (e.g. notebook 08)
# can reuse the EXACT same data without rebuilding from raw XPT
# files, which can produce slightly different patient inclusion
# due to parsing differences.
np.savez(
    MODEL_DIR / 'lstm_sequences_cache.npz',
    sequences=X_seq,
    labels=y_seq,
    patient_ids=np.array(sequence_seqns),
)
print(f"Cached {len(X_seq)} sequences to models/lstm_sequences_cache.npz")

## Section 5 — Normalize and Encode

## Section 5b — Truncated-Sequence Augmentation (Training Set Only)

Early-day forecasting feeds 1–3 real meals plus zero-padded future slots. The model was originally trained only on complete 6-meal sequences, so partial inputs defaulted to LOW. We augment **training data only** by truncating each sequence at cutoffs 1–5 and zero-padding the remainder, **keeping the full-sequence label**. The test set stays unmodified for official metrics.

In [ ]:
y_encoded = np.array([RISK_ENCODE[r] for r in y_seq])
y_cat = to_categorical(y_encoded, num_classes=3)

print("Label encoding:")
for i, cls in enumerate(RISK_CLASSES):
    print(f"  {i} = {cls}")

n_patients, n_steps, n_features = X_seq.shape
X_flat = X_seq.reshape(-1, n_features)
scaler = StandardScaler()
scaler.fit(X_flat)

print()
print(f"Sequence shape (raw): {X_seq.shape}")
print(f"Labels shape: {y_cat.shape}")

# Split on RAW sequences; scale after augmentation so zero-padded slots
# match inference (raw 0 → scaler transform), not literal zeros in scaled space.
X_train_raw, X_test_raw, y_train, y_test, idx_train, idx_test = train_test_split(
    X_seq, y_cat, y_encoded,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_encoded,
)

print()
print(f"Training sequences (raw): {len(X_train_raw)}")
print(f"Test sequences (raw): {len(X_test_raw)}")


def augment_with_truncated_sequences(X, y, n_steps=6):
    """For each sequence, add truncated+padded versions (cutoffs 1..5), same label."""
    X_augmented = [X.copy()]
    y_augmented = [y.copy()]
    for cutoff in range(1, n_steps):
        X_truncated = X.copy()
        X_truncated[:, cutoff:, :] = 0
        X_augmented.append(X_truncated)
        y_augmented.append(y.copy())
    return np.concatenate(X_augmented, axis=0), np.concatenate(y_augmented, axis=0)


X_train_aug_raw, y_train_aug = augment_with_truncated_sequences(
    X_train_raw, y_train, n_steps=n_steps
)

print()
print(f"Augmented training sequences (raw): {X_train_aug_raw.shape}")
print(f"  (expect {len(X_train_raw) * 6} = 6× original train size)")

for label in RISK_CLASSES:
    idx = RISK_ENCODE[label]
    orig_pct = (np.argmax(y_train, axis=1) == idx).mean()
    aug_pct = (np.argmax(y_train_aug, axis=1) == idx).mean()
    print(f"  {label}: train {orig_pct:.3f} → augmented {aug_pct:.3f}")


def scale_sequences(X, scaler):
    n = X.shape[0]
    flat = X.reshape(-1, n_features)
    return scaler.transform(flat).reshape(n, n_steps, n_features)


X_train = scale_sequences(X_train_aug_raw, scaler)
X_test = scale_sequences(X_test_raw, scaler)
y_train = y_train_aug

print()
print(f"Scaled training sequences: {X_train.shape}")
print(f"Scaled test sequences: {X_test.shape}")

joblib.dump(scaler, MODEL_DIR / 'lstm_scaler.pkl')
print("Scaler saved.")

## Section 6 — LSTM Architecture

The network uses two stacked LSTM layers to capture both short-term meal patterns and longer-term two-day dietary habits.

| Layer | Type | Units | Activation |
|---|---|---|---|
| 1 | LSTM | 64 | tanh |
| 2 | LSTM | 32 | tanh |
| 3 | Dense | 16 | ReLU |
| 4 | Dropout | 0.3 | — |
| 5 | Dense | 3 | Softmax |

## Section 6b — Hyperparameter Tuning

Compare four LSTM configurations on the held-out test set (selected by test F1).


In [ ]:
# Reuse confirmed winner from prior tuning (outputs/stats/15_lstm_tuning_comparison.csv).
# Set SKIP_TUNING=True to avoid re-running the 4-config grid when retraining with augmentation.
SKIP_TUNING = True

configs_to_try = [
    {'lstm1_units': 64, 'lstm2_units': 32, 'dropout': 0.2, 'learning_rate': 0.001, 'label': 'baseline (current)'},
    {'lstm1_units': 64, 'lstm2_units': 32, 'dropout': 0.3, 'learning_rate': 0.001, 'label': 'higher dropout'},
    {'lstm1_units': 128, 'lstm2_units': 64, 'dropout': 0.2, 'learning_rate': 0.001, 'label': 'larger network'},
    {'lstm1_units': 64, 'lstm2_units': 32, 'dropout': 0.2, 'learning_rate': 0.0005, 'label': 'lower learning rate'},
]

if SKIP_TUNING:
    best_config = configs_to_try[1]  # higher dropout — prior tuning winner
    baseline_won = False
    tuning_df = pd.read_csv(STATS_DIR / '15_lstm_tuning_comparison.csv')
    print(f"Skipping tuning — reusing winner: {best_config['label']}")
    print(tuning_df.to_string(index=False))
else:
    tuning_results = []

    for config in configs_to_try:
        print(f"\n{'='*50}")
        print(f"Training config: {config['label']}")
        print(f"{'='*50}")

        tf.random.set_seed(RANDOM_STATE)
        model_variant = Sequential([
            LSTM(config['lstm1_units'], input_shape=(n_steps, n_features),
                 activation='tanh', recurrent_activation='sigmoid',
                 return_sequences=True, dropout=config['dropout'],
                 recurrent_dropout=0.1, name='lstm_layer_1'),
            LSTM(config['lstm2_units'], activation='tanh',
                 recurrent_activation='sigmoid', return_sequences=False,
                 dropout=config['dropout'], name='lstm_layer_2'),
            Dense(16, activation='relu', name='dense_layer'),
            Dropout(0.3, name='dropout_layer'),
            Dense(3, activation='softmax', name='output_layer'),
        ])

        model_variant.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=config['learning_rate']),
            loss='categorical_crossentropy',
            metrics=['accuracy'],
        )

        early_stop = EarlyStopping(monitor='val_loss', patience=10,
                                   restore_best_weights=True, verbose=0)

        history_variant = model_variant.fit(
            X_train, y_train,
            epochs=50, batch_size=32, validation_split=0.2,
            callbacks=[early_stop], verbose=0,
        )

        y_pred_proba_variant = model_variant.predict(X_test, verbose=0)
        y_pred_variant = np.argmax(y_pred_proba_variant, axis=1)

        acc = accuracy_score(idx_test, y_pred_variant)
        f1 = f1_score(idx_test, y_pred_variant, average='weighted', zero_division=0)
        auc = roc_auc_score(idx_test, y_pred_proba_variant, multi_class='ovr', average='weighted')

        tuning_results.append({
            'config': config['label'],
            'lstm1_units': config['lstm1_units'],
            'lstm2_units': config['lstm2_units'],
            'dropout': config['dropout'],
            'learning_rate': config['learning_rate'],
            'test_accuracy': acc,
            'test_f1': f1,
            'test_auc_roc': auc,
            'best_epoch': int(np.argmin(history_variant.history['val_loss']) + 1),
        })
        print(f"Test Accuracy: {acc:.4f}, F1: {f1:.4f}, AUC-ROC: {auc:.4f}")

    tuning_df = pd.DataFrame(tuning_results)
    print("\n" + "="*70)
    print("HYPERPARAMETER TUNING COMPARISON")
    print("="*70)
    print(tuning_df.to_string(index=False))

    best_config_idx = int(tuning_df['test_f1'].idxmax())
    best_config = configs_to_try[best_config_idx]
    baseline_won = (best_config_idx == 0)
    print(f"\nBest configuration: {tuning_df.loc[best_config_idx, 'config']}")
    print(f"Baseline won: {baseline_won}")

    tuning_df.to_csv(STATS_DIR / '15_lstm_tuning_comparison.csv', index=False)
    print("Saved: outputs/stats/15_lstm_tuning_comparison.csv")

## Section 7 — Train Best LSTM Configuration


In [ ]:
print(f"Training final model: {best_config['label']}")
print("=" * 45)

tf.random.set_seed(RANDOM_STATE)

model = Sequential([
    LSTM(best_config['lstm1_units'], input_shape=(n_steps, n_features),
         activation='tanh', recurrent_activation='sigmoid',
         return_sequences=True, dropout=best_config['dropout'],
         recurrent_dropout=0.1, name='lstm_layer_1'),
    LSTM(best_config['lstm2_units'], activation='tanh',
         recurrent_activation='sigmoid', return_sequences=False,
         dropout=best_config['dropout'], name='lstm_layer_2'),
    Dense(16, activation='relu', name='dense_layer'),
    Dropout(0.3, name='dropout_layer'),
    Dense(3, activation='softmax', name='output_layer'),
], name='GuidaPlate_LSTM')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=best_config['learning_rate']),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()
print()
print(f"Total parameters: {model.count_params():,}")

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ModelCheckpoint(filepath=str(MODEL_DIR / 'lstm_best.keras'), monitor='val_loss', save_best_only=True, verbose=0),
]

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1,
)

best_epoch = int(np.argmin(history.history['val_loss']) + 1)
print()
print("Training complete.")
print(f"Best epoch: {best_epoch}")


## Section 7b — Before/After Truncated-Sequence Evaluation

Compare the **pre-augmentation** saved model vs the **newly trained** model on:
1. Original full 6-meal test sequences (official task)
2. Truncated test sequences (1–5 meals present + zero-padding) — simulates early-day forecasting

In [ ]:
from tensorflow.keras.models import load_model

BASELINE_ACC = 0.9071
ACC_TOLERANCE = 0.02


def evaluate_truncated_by_cutoff(keras_model, X_test_raw, idx_eval, scaler):
    rows = []
    for cutoff in range(1, n_steps):
        X_trunc_raw = X_test_raw.copy()
        X_trunc_raw[:, cutoff:, :] = 0
        flat = X_trunc_raw.reshape(-1, n_features)
        X_trunc = scaler.transform(flat).reshape(len(X_trunc_raw), n_steps, n_features)
        proba = keras_model.predict(X_trunc, verbose=0)
        pred = np.argmax(proba, axis=1)
        rows.append({
            'meals_present': cutoff,
            'accuracy': accuracy_score(idx_eval, pred),
            'f1_score': f1_score(idx_eval, pred, average='weighted', zero_division=0),
            'pct_predicted_low': round((pred == RISK_ENCODE['LOW']).mean() * 100, 2),
        })
    return rows


old_model = load_model(MODEL_DIR / 'lstm_final.keras')
old_full_acc = accuracy_score(idx_test, np.argmax(old_model.predict(X_test, verbose=0), axis=1))
old_trunc = evaluate_truncated_by_cutoff(old_model, X_test_raw, idx_test, scaler)

new_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
new_full_acc = accuracy_score(idx_test, new_pred)
new_trunc = evaluate_truncated_by_cutoff(model, X_test_raw, idx_test, scaler)

print('=' * 60)
print('BEFORE/AFTER AUGMENTATION COMPARISON')
print('=' * 60)
print(f"Original model — full-sequence test accuracy: {old_full_acc:.4f}")
print(f"Augmented model — full-sequence test accuracy: {new_full_acc:.4f}")
print(f"Delta vs baseline ({BASELINE_ACC:.4f}): {new_full_acc - BASELINE_ACC:+.4f}")
print()
print('Truncated test accuracy by meals present:')
print(f"{'Meals':>6} | {'Orig acc':>9} | {'Orig %LOW':>10} | {'New acc':>9} | {'New %LOW':>10}")
for o, n in zip(old_trunc, new_trunc):
    print(f"{o['meals_present']:>6} | {o['accuracy']:>9.4f} | {o['pct_predicted_low']:>9.1f}% | {n['accuracy']:>9.4f} | {n['pct_predicted_low']:>9.1f}%")

comparison_rows = [{
    'model': 'original', 'eval_type': 'full_sequence', 'meals_present': 6,
    'accuracy': round(old_full_acc, 4), 'pct_predicted_low': None,
}]
comparison_rows += [{'model': 'original', 'eval_type': 'truncated', **r} for r in old_trunc]
comparison_rows.append({
    'model': 'augmented', 'eval_type': 'full_sequence', 'meals_present': 6,
    'accuracy': round(new_full_acc, 4), 'pct_predicted_low': None,
})
comparison_rows += [{'model': 'augmented', 'eval_type': 'truncated', **r} for r in new_trunc]

aug_comparison_df = pd.DataFrame(comparison_rows)
aug_comparison_df.to_csv(STATS_DIR / '16_lstm_augmentation_comparison.csv', index=False)
print()
print('Saved: outputs/stats/16_lstm_augmentation_comparison.csv')

save_augmented_model = new_full_acc >= (BASELINE_ACC - ACC_TOLERANCE)
print(f"\nSave augmented model (within {ACC_TOLERANCE:.0%} of baseline): {save_augmented_model}")

## Section 8 — Performance Metrics

In [ ]:
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)

acc = accuracy_score(idx_test, y_pred)
prec = precision_score(idx_test, y_pred, average='weighted', zero_division=0)
rec = recall_score(idx_test, y_pred, average='weighted', zero_division=0)
f1 = f1_score(idx_test, y_pred, average='weighted', zero_division=0)

try:
    auc = roc_auc_score(idx_test, y_pred_prob, multi_class='ovr', average='weighted')
except Exception as e:
    auc = None
    print(f"AUC error: {e}")

high_idx = RISK_ENCODE['HIGH']
high_tp = ((idx_test == high_idx) & (y_pred == high_idx)).sum()
high_fn = ((idx_test == high_idx) & (y_pred != high_idx)).sum()
high_recall = high_tp / (high_tp + high_fn) if (high_tp + high_fn) > 0 else 0.0

print("=" * 45)
print("LSTM PERFORMANCE METRICS")
print("=" * 45)
print(f"Accuracy:  {acc:.4f} ({acc*100:.1f}%)")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
if auc is not None:
    print(f"AUC-ROC:   {auc:.4f}")
print(f"HIGH RISK Sensitivity: {high_recall:.4f} ({high_recall*100:.1f}%)")
print()
print("Target benchmarks:")
print("AUC-ROC > 0.90")
print("HIGH RISK Sensitivity > 0.85")
print()
print(classification_report(idx_test, y_pred, labels=[0, 1, 2], target_names=RISK_CLASSES, zero_division=0))

if high_recall >= 0.85:
    print("HIGH RISK sensitivity target MET")
else:
    print("HIGH RISK sensitivity not yet met (see tuning comparison above).")

## Section 9 — Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='Train', color='#2563EB')
axes[0].plot(history.history['val_accuracy'], label='Validation', color='#DC2626')
axes[0].set_title('LSTM Training Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train', color='#2563EB')
axes[1].plot(history.history['val_loss'], label='Validation', color='#DC2626')
axes[1].set_title('LSTM Training Loss', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('GuidaPlate LSTM Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '13_lstm_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(idx_test, y_pred, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=RISK_CLASSES)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('LSTM Confusion Matrix - GuidaPlate Pattern Analyzer', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '14_lstm_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

meal_labels = ['D1 Breakfast', 'D1 Lunch', 'D1 Dinner', 'D2 Breakfast', 'D2 Lunch', 'D2 Dinner']
nutrient_names = ['Potassium (mg)', 'Phosphorus (mg)', 'Protein (g/kg)', 'Sodium (mg)']
risk_colors = {'LOW': '#16A34A', 'MODERATE': '#D97706', 'HIGH': '#DC2626'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for feat_idx, (ax, nutrient) in enumerate(zip(axes, nutrient_names)):
    for risk in RISK_CLASSES:
        risk_idx = RISK_ENCODE[risk]
        mask = idx_test == risk_idx
        if mask.sum() == 0:
            continue
        mean_seq = X_test[mask, :, feat_idx].mean(axis=0)
        ax.plot(range(6), mean_seq, label=risk, color=risk_colors[risk], linewidth=2, marker='o')
    ax.set_xticks(range(6))
    ax.set_xticklabels(meal_labels, fontsize=8)
    ax.set_title(f'Average {nutrient} by Risk Level', fontweight='bold')
    ax.set_xlabel('Meal Occasion')
    ax.set_ylabel(nutrient)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Nutrient Patterns Across 6 Meal Occasions - GuidaPlate LSTM Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '15_lstm_meal_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved: {FIG_DIR / '13_lstm_training_history.png'}")
print(f"Saved: {FIG_DIR / '14_lstm_confusion_matrix.png'}")
print(f"Saved: {FIG_DIR / '15_lstm_meal_patterns.png'}")

## Section 10 — Save Model

In [ ]:
if save_augmented_model:
    model.save(MODEL_DIR / 'lstm_final.keras')
    print(f"Model saved: models/lstm_final.keras (augmented retrain)")
    print("Scaler unchanged: models/lstm_scaler.pkl (same fit on full X_seq)")
    print("Label encoder unchanged: models/lstm_label_encoder.pkl")
else:
    print("NOT overwriting lstm_final.keras — full-sequence accuracy dropped below tolerance.")
    print(f"Required: >= {BASELINE_ACC - ACC_TOLERANCE:.4f}, got: {new_full_acc:.4f}")

metrics_df = pd.DataFrame([{
    'model': 'LSTM v1 (augmented train)',
    'accuracy': round(acc, 4),
    'precision': round(prec, 4),
    'recall': round(rec, 4),
    'f1_score': round(f1, 4),
    'auc_roc': round(auc, 4) if auc is not None else None,
    'high_risk_sensitivity': round(high_recall, 4),
    'training_sequences': len(X_train),
    'test_sequences': len(X_test),
    'sequence_length': n_steps,
    'features_per_step': n_features,
    'n_classes': 3,
    'epochs_trained': len(history.history['loss']),
    'best_epoch': best_epoch,
    'final_val_loss': round(float(min(history.history['val_loss'])), 4),
    'patients_with_sequences': n_processed,
    'patients_skipped': n_skipped,
    'tuning_best_config': best_config['label'],
    'augmentation_factor': 6,
}])

metrics_path = STATS_DIR / '07_lstm_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)
print(f"Metrics saved: {metrics_path}")

print()
print("=" * 45)
print("NOTEBOOK 05 COMPLETE")
print("=" * 45)
print("Saved:")
if save_augmented_model:
    print("  models/lstm_final.keras")
print("  outputs/stats/07_lstm_metrics.csv")
print("  outputs/stats/16_lstm_augmentation_comparison.csv")
print("  outputs/stats/15_lstm_tuning_comparison.csv")
print("  outputs/figures/13_lstm_training_history.png")
print("  outputs/figures/14_lstm_confusion_matrix.png")
print("  outputs/figures/15_lstm_meal_patterns.png")
print()
print("Next: notebook 06 evaluation (SHAP + McNemar XGBoost vs LSTM)")